In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import sys
sys.path.append("..")

import sys
sys.path.append("..")

from plotting_tools import * 

In [ ]:
%config InlineBackend.print_figure_kwargs = {'dpi': 110, 'bbox_inches': 'tight'}

In [ ]:
experiment_series ="pop8"
precipitation_setting = "Rain"
decay_setting = "decay"

plot_path = f"../../plots/Figure_5"
os.makedirs(plot_path, exist_ok=True)

In [ ]:
# Load the data
result_path = f"../../../preprocessing/preprocessed_data/{experiment_series}"

df = pd.read_csv(f"{result_path}/substances/{decay_setting}_{precipitation_setting}_output_scaled.csv")
df["Date"] = pd.to_datetime(start_date) + pd.to_timedelta(df["time_in_minutes"], unit="min")

In [ ]:
df_base = pd.read_csv(f"{result_path}/substances/{"no_decay"}_{"noRain"}_output_scaled.csv")
df_base["Date"] = pd.to_datetime(start_date) + pd.to_timedelta(df_base["time_in_minutes"], unit="min")
df_base = df_base[df_base.variable=="COV19"]
df_base["location"] = df_base.manhole.apply(lambda x: manhole_names[x] if x in manhole_names else x)

In [ ]:
df = df.loc[df.variable=="COV19"]
df["location"] = df.manhole.apply(lambda x: manhole_names[x] if x in manhole_names else x)

In [ ]:
def simulate_sampling_strategies_legacy(df_COV_sim):
    df = df_COV_sim.copy()
    df["month"] = [el.month for el in df["Date"]]
    df["day"] = [el.day for el in df["Date"]]
    df["hour"] = [el.hour for el in df["Date"]]

    # get hourly measurements
    df = df.loc[df.time_in_minutes.mod(60) == 0, :]

    # calculate 24h samples
    df_24h = df.groupby(["location", "simulation_id", "day", "month"])[["value", "hour", "time_in_days", "Date"]].mean().reset_index().rename(columns={"value": "24h_sample"})
    # only consider 24h samples with 24h of data (i.e. remove first and last day of observations)
    df_24h = df_24h.loc[df_24h.hour == 11.5, ["location", "simulation_id", "time_in_days", "day", "24h_sample", "Date"]]
    df_24h["time_in_days"] = df_24h["time_in_days"] + 1.5 / 24  # plot at 14:00

    # extract morning sample column
    df_morning_sample = df.loc[df.hour == 10, :].rename(columns={"value": "morning_sample"}).loc[:, ["Date", "time_in_days", "location", "simulation_id", "morning_sample"]]

    # combine everything
    df_measurements = pd.merge(df, pd.merge(df_24h, df_morning_sample, on=["location", "simulation_id", "Date", "time_in_days"], how="outer"), on=["location", "simulation_id", "Date", "time_in_days"], how="outer")
    df_measurements.rename({"value": "COVID19"}, axis=1, inplace=True)
    return df_measurements


def _report_missing_values(df_base, df_sampled, expected_count, strategy_name, grouping_cols=None):
    if grouping_cols is None:
        grouping_cols = ["location", "simulation_id", "day", "month"]

    base_groups = df_base.groupby(grouping_cols).size().rename("n_base").reset_index()[grouping_cols]
    sampled_counts = df_sampled.groupby(grouping_cols).size().rename("n_sampled").reset_index()

    missing_df = base_groups.merge(sampled_counts, on=grouping_cols, how="left")
    missing_df["n_sampled"] = missing_df["n_sampled"].fillna(0)
    missing_df["missing_count"] = expected_count - missing_df["n_sampled"]
    missing_df = missing_df.loc[missing_df["missing_count"] > 0, :]

    if missing_df.empty:
        print(f"[{strategy_name}] No missing values detected.")
    else:
        total_missing = int(missing_df["missing_count"].sum())
        print(f"[{strategy_name}] Missing {total_missing} expected values across {len(missing_df)} groups.")


def _compound_sample(df_sampled, expected_count, output_column, grouping_cols=None):
    if grouping_cols is None:
        grouping_cols = ["location", "simulation_id", "day", "month"]

    sample_values = (
        df_sampled.groupby(grouping_cols)[["value", "time_in_days", "Date"]]
        .mean()
        .reset_index()
    )
    sample_counts = (
        df_sampled.groupby(grouping_cols)
        .size()
        .rename("n_obs")
        .reset_index()
    )

    sample_values = sample_values.merge(sample_counts, on=grouping_cols, how="left")
    sample_values = sample_values.loc[sample_values.n_obs == expected_count, ["Date", "time_in_days", "location", "simulation_id", "value"]]
    sample_values = sample_values.rename(columns={"value": output_column})

    return sample_values


def simulate_sampling_strategies(df_COV_sim):
    df = df_COV_sim.copy()
    df["month"] = [el.month for el in df["Date"]]
    df["day"] = [el.day for el in df["Date"]]
    df["hour"] = [el.hour for el in df["Date"]]

    # keep hourly values for legacy-compatible strategies
    df_hourly = df.loc[df.time_in_minutes.mod(60) == 0, :].copy()

    # legacy 24h sample (1h resolution over 24h)
    df_24h = df_hourly.groupby(["location", "simulation_id", "day", "month"])[["value", "hour", "time_in_days", "Date"]].mean().reset_index().rename(columns={"value": "24h_sample"})
    df_24h = df_24h.loc[df_24h.hour == 11.5, ["location", "simulation_id", "time_in_days", "day", "24h_sample", "Date"]]
    df_24h["time_in_days"] = df_24h["time_in_days"] + 1.5 / 24

    # legacy morning grab sample at 10:00
    df_morning_sample = df_hourly.loc[df_hourly.hour == 10, :].rename(columns={"value": "morning_sample"}).loc[:, ["Date", "time_in_days", "location", "simulation_id", "morning_sample"]]

    # 24h compound sample at 15-minute resolution
    df_15min = df.loc[df.time_in_minutes.mod(15) == 0, :].copy()
    _report_missing_values(df, df_15min, expected_count=96, strategy_name="24h_compound_15min")
    df_15min_24h = _compound_sample(df_15min, expected_count=96, output_column="sample_15min_24h")

    # 24h compound sample at 30-minute resolution
    df_30min = df.loc[df.time_in_minutes.mod(30) == 0, :].copy()
    _report_missing_values(df, df_30min, expected_count=48, strategy_name="24h_compound_30min")
    df_30min_24h = _compound_sample(df_30min, expected_count=48, output_column="sample_30min_24h")

    # evening grab sample at 19:00
    df_evening_sample = df_hourly.loc[df_hourly.hour == 19, :].copy()
    _report_missing_values(df_hourly, df_evening_sample, expected_count=1, strategy_name="grab_sample_19h")
    df_evening_sample = df_evening_sample.rename(columns={"value": "evening_sample_19"}).loc[:, ["Date", "time_in_days", "location", "simulation_id", "evening_sample_19"]]

    # 4h compound samples at hourly resolution (all six 4h blocks per day)
    df_4h_hourly = df_hourly.copy()
    df_4h_hourly["block_4h"] = (df_4h_hourly["hour"] // 4).astype(int)
    grouping_cols_4h = ["location", "simulation_id", "day", "month", "block_4h"]
    _report_missing_values(
        df_4h_hourly,
        df_4h_hourly,
        expected_count=4,
        strategy_name="4h_compound_hourly",
        grouping_cols=grouping_cols_4h,
    )
    df_4h_hourly = _compound_sample(
        df_4h_hourly,
        expected_count=4,
        output_column="sample_4h_hourly",
        grouping_cols=grouping_cols_4h,
    )

    sampling_frames = [df_24h, df_morning_sample, df_15min_24h, df_30min_24h, df_evening_sample, df_4h_hourly]
    df_sampling_all = sampling_frames[0]
    for df_sampling_curr in sampling_frames[1:]:
        df_sampling_all = pd.merge(
            df_sampling_all,
            df_sampling_curr,
            on=["location", "simulation_id", "Date", "time_in_days"],
            how="outer",
        )

    df_measurements = pd.merge(df_hourly, df_sampling_all, on=["location", "simulation_id", "Date", "time_in_days"], how="outer")
    df_measurements.rename({"value": "COVID19"}, axis=1, inplace=True)
    return df_measurements


In [ ]:
df_measurements = simulate_sampling_strategies(df)
df_sampling = pd.melt(
    df_measurements,
    id_vars=["Date", "location", "simulation_id"],
    value_vars=[
        "COVID19",
        "morning_sample",
        "24h_sample",
        "sample_15min_24h",
        "sample_30min_24h",
        "evening_sample_19",
        "sample_4h_hourly",
    ],
).dropna()


### Legacy Compatibility Check
Ensure the legacy 24-hour (1h resolution) and morning grab sample outputs are unchanged.

In [ ]:
legacy_measurements = simulate_sampling_strategies_legacy(df)

for col in ["24h_sample", "morning_sample"]:
    old_values = (
        legacy_measurements[["Date", "location", "simulation_id", col]]
        .dropna()
        .sort_values(["Date", "location", "simulation_id"])
        .reset_index(drop=True)
    )
    new_values = (
        df_measurements[["Date", "location", "simulation_id", col]]
        .dropna()
        .sort_values(["Date", "location", "simulation_id"])
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(old_values, new_values, check_dtype=False)

print("Legacy compatibility check passed for 24h_sample and morning_sample.")


### Metrics

In [ ]:
df_sampling.Date.describe()

In [ ]:
location = "S_M2"

In [ ]:
def get_metrics_24h_sample(location, min_date="2020-03-02", max_date="2020-06-02"):
    metric_24h_sample = df_sampling.loc[df_sampling.variable.isin(["24h_sample"]), ["Date", "location", "simulation_id", "value", "variable"]].copy()
    metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.location==location]
    df_base_sub = df_base.loc[df_base.variable=="COV19", ["Date", "location", "simulation_id", "value", "variable"]].copy()
    df_base_sub = df_base_sub.loc[df_base_sub.location==location]
    metric_24h_sample = pd.concat([metric_24h_sample, df_base_sub])

    metric_24h_sample = (
        metric_24h_sample
        .pivot(index=["Date", "simulation_id"], columns="variable", values="value")
        .reset_index()
        .sort_values(["simulation_id", "Date"])
    )

    # --- interpolate first (per simulation) ---
    metric_24h_sample["24h_sample"] = (
        metric_24h_sample
        .groupby("simulation_id", group_keys=False)["24h_sample"]
        .apply(lambda s: s.interpolate(method="linear"))
    )

    # ensure interpolation is only within observation window
    cutoff = pd.Timestamp("2020-06-01 11:30")
    metric_24h_sample.loc[metric_24h_sample["Date"] > cutoff, "24h_sample"] = np.nan

    # keep on-the-hour stamps before averaging
    #on_the_hour = metric_24h_sample.loc[metric_24h_sample["Date"].dt.minute == 0].copy()

    # --- then average across simulations ---
    metric_24h_sample = (
        metric_24h_sample
        .groupby("Date")[["24h_sample", "COV19"]]
        .mean()
        .sort_index()
        .reset_index()
    )

    metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.Date >= min_date]
    metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.Date <= max_date]

    # MAE on the averaged series
    metric_24h_sample["MAE"] = (metric_24h_sample["24h_sample"] - metric_24h_sample["COV19"]).abs()

    return metric_24h_sample

def get_metric_morning_sample(location, min_date="2020-03-02", max_date="2020-06-02"):
    metric_morning_sample = df_sampling.loc[df_sampling.variable.isin(["morning_sample"]), ["Date", "location", "simulation_id", "value", "variable"]].copy()
    metric_morning_sample = metric_morning_sample.loc[metric_morning_sample.location==location]
    df_base_sub = df_base.loc[df_base.variable=="COV19", ["Date", "location", "simulation_id", "value", "variable"]].copy()
    df_base_sub = df_base_sub.loc[df_base_sub.location==location]
    metric_morning_sample = pd.concat([metric_morning_sample, df_base_sub])

    metric_morning_sample = (
        metric_morning_sample
        .pivot(index=["Date", "simulation_id"], columns="variable", values="value")
        .reset_index()
        .sort_values(["simulation_id", "Date"])
    )

    # --- interpolate first (per simulation) ---
    metric_morning_sample["morning_sample"] = (
        metric_morning_sample
        .groupby("simulation_id", group_keys=False)["morning_sample"]
        .apply(lambda s: s.interpolate(method="linear"))
    )

    # ensure interpolation is only within observation window
    cutoff = pd.Timestamp("2020-06-01 10:00")
    metric_morning_sample.loc[metric_morning_sample["Date"] > cutoff, "morning_sample"] = np.nan

    # keep on-the-hour stamps before averaging
    #on_the_hour = metric_morning_sample.loc[metric_morning_sample["Date"].dt.minute == 0].copy()

    # --- then average across simulations ---
    metric_morning_sample = (
        metric_morning_sample
        .groupby("Date")[["morning_sample", "COV19"]]
        .mean()
        .sort_index()
        .reset_index()
    )

    metric_morning_sample = metric_morning_sample.loc[metric_morning_sample.Date >= min_date]
    metric_morning_sample = metric_morning_sample.loc[metric_morning_sample.Date <= max_date]

    # MAE on the averaged series
    metric_morning_sample["MAE"] = (metric_morning_sample["morning_sample"] - metric_morning_sample["COV19"]).abs()

    # metric_morning_sample.describe()
    return metric_morning_sample


In [ ]:
def get_mse_metrics_24h_sample(location, min_date="2020-03-02", max_date="2020-06-02"):
    metric_24h_sample = df_sampling.loc[df_sampling.variable.isin(["24h_sample"]), ["Date", "location", "simulation_id", "value", "variable"]].copy()
    metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.location==location]
    df_base_sub = df_base.loc[df_base.variable=="COV19", ["Date", "location", "simulation_id", "value", "variable"]].copy()
    df_base_sub = df_base_sub.loc[df_base_sub.location==location]
    metric_24h_sample = pd.concat([metric_24h_sample, df_base_sub])

    metric_24h_sample = (
        metric_24h_sample
        .pivot(index=["Date", "simulation_id"], columns="variable", values="value")
        .reset_index()
        .sort_values(["simulation_id", "Date"])
    )

    # --- interpolate first (per simulation) ---
    metric_24h_sample["24h_sample"] = (
        metric_24h_sample
        .groupby("simulation_id", group_keys=False)["24h_sample"]
        .apply(lambda s: s.interpolate(method="linear"))
    )

    # ensure interpolation is only within observation window
    cutoff = pd.Timestamp("2020-06-01 11:30")
    metric_24h_sample.loc[metric_24h_sample["Date"] > cutoff, "24h_sample"] = np.nan

    # keep on-the-hour stamps before averaging
    #on_the_hour = metric_24h_sample.loc[metric_24h_sample["Date"].dt.minute == 0].copy()

    # --- then average across simulations ---
    metric_24h_sample = (
        metric_24h_sample
        .groupby("Date")[["24h_sample", "COV19"]]
        .mean()
        .sort_index()
        .reset_index()
    )

    metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.Date >= min_date]
    metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.Date <= max_date]

    # MAE on the averaged series
    metric_24h_sample["MSE"] = (metric_24h_sample["24h_sample"] - metric_24h_sample["COV19"])**2

    return metric_24h_sample

def get_mse_metric_morning_sample(location, min_date="2020-03-02", max_date="2020-06-02"):
    metric_morning_sample = df_sampling.loc[df_sampling.variable.isin(["morning_sample"]), ["Date", "location", "simulation_id", "value", "variable"]].copy()
    metric_morning_sample = metric_morning_sample.loc[metric_morning_sample.location==location]
    df_base_sub = df_base.loc[df_base.variable=="COV19", ["Date", "location", "simulation_id", "value", "variable"]].copy()
    df_base_sub = df_base_sub.loc[df_base_sub.location==location]
    metric_morning_sample = pd.concat([metric_morning_sample, df_base_sub])

    metric_morning_sample = (
        metric_morning_sample
        .pivot(index=["Date", "simulation_id"], columns="variable", values="value")
        .reset_index()
        .sort_values(["simulation_id", "Date"])
    )

    # --- interpolate first (per simulation) ---
    metric_morning_sample["morning_sample"] = (
        metric_morning_sample
        .groupby("simulation_id", group_keys=False)["morning_sample"]
        .apply(lambda s: s.interpolate(method="linear"))
    )

    # ensure interpolation is only within observation window
    cutoff = pd.Timestamp("2020-06-01 10:00")
    metric_morning_sample.loc[metric_morning_sample["Date"] > cutoff, "morning_sample"] = np.nan

    # keep on-the-hour stamps before averaging
    #on_the_hour = metric_morning_sample.loc[metric_morning_sample["Date"].dt.minute == 0].copy()

    # --- then average across simulations ---
    metric_morning_sample = (
        metric_morning_sample
        .groupby("Date")[["morning_sample", "COV19"]]
        .mean()
        .sort_index()
        .reset_index()
    )

    metric_morning_sample = metric_morning_sample.loc[metric_morning_sample.Date >= min_date]
    metric_morning_sample = metric_morning_sample.loc[metric_morning_sample.Date <= max_date]

    # MAE on the averaged series
    metric_morning_sample["MSE"] = (metric_morning_sample["morning_sample"] - metric_morning_sample["COV19"])**2
    # metric_morning_sample.describe()
    return metric_morning_sample


In [ ]:
MSE_all = pd.DataFrame()
MAE_all = pd.DataFrame()
for location in df_sampling.location.unique():
    metric_24h_sample = get_mse_metrics_24h_sample(location)
    metric_morning_sample = get_mse_metric_morning_sample(location)
    MSE_df = pd.DataFrame({"morning_sample": metric_morning_sample["MSE"], "24h_sample": metric_24h_sample["MSE"]})
    MSE_df_d = MSE_df.describe().reset_index()
    MSE_df_d["location"] = location
    MSE_all = pd.concat([MSE_all, MSE_df_d], ignore_index=True)
    metric_24h_sample = get_metrics_24h_sample(location)
    metric_morning_sample = get_metric_morning_sample(location)
    MAE_df = pd.DataFrame({"morning_sample": metric_morning_sample["MAE"], "24h_sample": metric_24h_sample["MAE"]})
    MAE_df_d = MAE_df.describe().reset_index()
    MAE_df_d["location"] = location
    MAE_all = pd.concat([MAE_all, MAE_df_d], ignore_index=True)

    print(location)
    """
    fig, ax = plt.subplots(figsize = (6,3), dpi=300)
    sns.boxplot(data=MAE_df.loc[(MAE_df["24h_sample"].notna())&(MAE_df["morning_sample"].notna())], linewidth=1.5, palette=[red, teal])
    ax.set_ylabel(f"Absolute error")
    ax.set_xticklabels(["Daily\ngrab sample", "24-hour\ncompound sample"], size=15)
    ax.tick_params(axis='y')
    ax.set_yscale('log')
    plt.tight_layout()
    fig.savefig(f"{plot_path}/sampling_strategy_{location}_mae.png", bbox_inches="tight")
    """    


In [ ]:
pop_size = {'N_Ua': 11226,
                   'N_Ub': 3028,
                   'N_Uc': 11719,
                   'C_U': 11866,
                   'S_Ua': 27636,
                   'S_Ub': 28353,
                   'S_M1': 214728,
                   'S_M2': 554321,
                   'S_M3': 644436,
                   'S_M4':940939,
                   'E_U': 44871,
                   'E_M': 138107,
                   'N_D': 205406,
                   'SCE_D1': 1331350,
                   'SCE_D2': 1331350,
                   'Overall': 1536756}

pop_density = {'N_Ua': 2789.08,
                   'N_Ub': 2752.98,
                   'N_Uc': 1058.44,
                   'C_U': 3390.09,
                   'S_Ua': 5657.83,
                   'S_Ub': 4467.15,
                   'S_M1': 6157.57,
                   'S_M2': 7566.13,
                   'S_M3': 8130.24,
                   'S_M4': 8067.55,
                   'E_U': 7644.38,
                   'E_M': 3774.66,
                   'N_D': 2229.19,
                   'SCE_D1': 6281.30,
                   'SCE_D2': 6281.30,
                   'Overall': 5053.48}

MAE_all["pop_size"] = MAE_all["location"].map(pop_size)
MAE_all["pop_density"] = MAE_all["location"].map(pop_density)

MSE_all["pop_size"] = MSE_all["location"].map(pop_size)
MSE_all["pop_density"] = MSE_all["location"].map(pop_density)

In [ ]:
# MSE diff (left y-axis)
df_mse = MSE_all.loc[MSE_all["index"] == "mean"].sort_values("pop_size")
mse_diff = df_mse["morning_sample"] - df_mse["24h_sample"]

# MAE diff (right y-axis)
df_mae = MAE_all.loc[MAE_all["index"] == "mean"].sort_values("pop_size")
mae_diff = df_mae["morning_sample"] - df_mae["24h_sample"]

fig, ax1 = plt.subplots(figsize=(6, 3))

# Left axis: MSE
s1 = ax1.scatter(df_mse["pop_size"], mse_diff, label="ΔMSE", color=black, zorder=20)
ax1.set_xlabel("Upstream citizens [#]")
ax1.set_ylabel("Avg. ΔMSE\n(grab - 24h)")
#ax1.spines["left"].set_color(red)
# ax1.tick_params(axis="y", colors=red)
ax1.set_xscale('log')

# Right axis: MAE
ax2 = ax1.twinx()
s2 = ax2.scatter(df_mae["pop_size"], mae_diff, marker="d", label="ΔMAE", color=dark_grey, zorder=20)
ax2.set_ylabel("Avg. ΔMAE\n(grab - 24h)")
#ax2.spines["right"].set_color(teal)
# ax2.tick_params(axis="y", colors=teal)
ax2.set_xscale('log')

# --- Make twin axis background transparent so shading shows through ---
ax2.patch.set_visible(False)

# --- Center both y-axes around zero ---
ymax1 = max(abs(mse_diff).max(), abs(mse_diff).min())
ax1.set_ylim(-ymax1*1.05, ymax1*1.05)   

ymax2 = max(abs(mae_diff).max(), abs(mae_diff).min())
ax2.set_ylim(-ymax2*1.05, ymax2*1.05)

# --- Shading for positive (>0) and negative (<0) regions ---
ylim1_low, ylim1_high = ax1.get_ylim()
ax1.axhspan(0, ylim1_high, facecolor=tan, alpha=0.7, zorder=0)
ax1.axhspan(ylim1_low, 0, facecolor=teal, alpha=0.7, zorder=0)

# --- Reference lines ---
#ax1.axhline(0, color=dark_grey, linestyle="--", linewidth=1)   # horizontal y=0

# Combined legend
handles = [s1, s2]
labels = [h.get_label() for h in handles]
ax1.legend(handles, labels, loc="best", framealpha=0.15)

plt.tight_layout()
plt.show()
# fig.savefig(f"{plot_path}/sampling_strategy_mse_mae_difference_vs_population.png", bbox_inches="tight", dpi=300)


### Boxplot of Metrics

In [ ]:
location = "S_M2"

In [ ]:
metric_24h_sample = get_metrics_24h_sample(location)
metric_morning_sample = get_metric_morning_sample(location)
MAE_df = pd.DataFrame({"morning_sample": metric_morning_sample["MAE"], "24h_sample": metric_24h_sample["MAE"]})

In [ ]:
fig, ax = plt.subplots(figsize = (6,3), dpi=300)
sns.boxplot(data=MAE_df.loc[(MAE_df["24h_sample"].notna())&(MAE_df["morning_sample"].notna())], linewidth=1.5, palette=[red, teal])
means = MAE_df.mean()
ax.scatter(
    x=np.arange(len(means)), 
    y=means,
    marker='^',      # Triangle marker
    color='black',   # Marker colour
    s=50,            # Marker size
    zorder=3         # Draw above boxplot
)

ax.set_ylabel(f"Absolute error")
ax.set_xticklabels(["Daily\ngrab sample", "24-hour\ncompound sample"], size=15)
ax.tick_params(axis='y')
ax.set_yscale('log')
plt.tight_layout()

##### comparison with same decay/rain setting

In [ ]:
metric_24h_sample = df_sampling.loc[df_sampling.variable.isin(["24h_sample", "COVID19"]), ["Date", "location", "simulation_id", "value", "variable"]].copy()
metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.location==location]

In [ ]:
metric_24h_sample = df_sampling.loc[df_sampling.variable.isin(["24h_sample", "COVID19"]), ["Date", "location", "simulation_id", "value", "variable"]].copy()
metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.location==location]
metric_24h_sample = metric_24h_sample.pivot(index=["Date", "simulation_id"], columns="variable", values="value").reset_index()

# take average across simulations
metric_24h_sample = metric_24h_sample.groupby("Date")[["24h_sample", "COVID19"]].mean().sort_index().reset_index()

# interpolate linearly between 24h samples
metric_24h_sample["24h_sample"] = metric_24h_sample["24h_sample"].interpolate(method="linear")
# ensure that linear interpolation happens only between "observations"
metric_24h_sample.loc[metric_24h_sample.Date>"2020-06-01 11:30", "24h_sample"] = np.nan
metric_24h_sample["minute"] = metric_24h_sample["Date"].dt.minute
metric_24h_sample = metric_24h_sample.loc[metric_24h_sample.minute==0]

metric_24h_sample["MAE"] = np.abs(metric_24h_sample["24h_sample"] - metric_24h_sample["COVID19"])
metric_24h_sample.describe()

In [ ]:
metric_morning_sample = df_sampling.loc[df_sampling.variable.isin(["morning_sample", "COVID19"]), ["Date", "location", "simulation_id", "value", "variable"]].copy()
metric_morning_sample = metric_morning_sample.loc[metric_morning_sample.location==location]
metric_morning_sample = metric_morning_sample.pivot(index=["Date", "simulation_id"], columns="variable", values="value").reset_index()

# take average across simulations
metric_morning_sample = metric_morning_sample.groupby("Date")[["morning_sample", "COVID19"]].mean().sort_index().reset_index()

# interpolate linearly between 24h samples
metric_morning_sample["morning_sample"] = metric_morning_sample["morning_sample"].interpolate(method="linear")
# ensure that linear interpolation happens only between "observations"
metric_morning_sample.loc[metric_morning_sample.Date>"2020-06-02 10:00", "morning_sample"] = np.nan

metric_morning_sample["MAE"] = np.abs(metric_morning_sample["morning_sample"] - metric_morning_sample["COVID19"])
metric_morning_sample.describe()

In [ ]:
MAE_df = pd.DataFrame({"morning_sample": metric_morning_sample["MAE"], "24h_sample": metric_24h_sample["MAE"]})
MAE_df.describe()

In [ ]:
fig, ax = plt.subplots(figsize = (6,3), dpi=300)
sns.boxplot(data=MAE_df, linewidth=1.5, palette=[red, teal])
ax.set_ylabel(f"Absolute error")
ax.set_xticklabels(["Daily\ngrab sample", "24-hour\ncompound sample"], size=15)
ax.tick_params(axis='y')
ax.set_yscale('log')
plt.tight_layout()

### Plot Timeseries

In [ ]:
def plot_sampling_strategy(station):
    df_curr = df_sampling.loc[df_sampling.location==station, :]
    df_curr = df_curr.loc[(df_curr.Date>"2020-03-08 08:00:00")&(df_curr.Date<"2020-03-14 15:00:00")]
    #df_curr = df_curr.loc[(df_curr.Date>"2020-03-08 08:00:00")&(df_curr.Date<"2020-04-14 15:00:00")]
    #df_curr = df_curr.loc[(df_curr.Date>"2020-04-08 08:00:00")&(df_curr.Date<"2020-06-14 15:00:00")]

    # Calculate and add 90percentiles as error bars
    ci_morning = df_curr[df_curr.variable == "morning_sample"].groupby("Date")['value'].agg([lambda x: np.mean(x), lambda x: np.percentile(x, 5), lambda x: np.percentile(x, 95)])
    ci_morning.columns = ['mean', 'p5', 'p95']

    ci_24h = df_curr[df_curr.variable == "24h_sample"].groupby("Date")['value'].agg([lambda x: np.mean(x), lambda x: np.percentile(x, 5), lambda x: np.percentile(x, 95)])
    ci_24h.columns = ['mean', 'p5', 'p95']


    # Use Seaborn's default 95% confidence interval with error bars
    fig, ax = plt.subplots(figsize=(7, 2.7), dpi=300) 
    # fig, ax = plt.subplots(figsize=(17, 2.7), dpi=300) 
    

    ax2 = ax.twinx()
    sns.lineplot(df_curr.loc[df_curr.variable == "morning_sample", :], ax=ax2, x="Date", y="value", errorbar=None, color=teal, marker="o", label="Daily grab sample")
    sns.lineplot(df_curr.loc[df_curr.variable == "24h_sample", :], ax=ax2, x="Date", y="value", errorbar=None, color=tan, marker="o", label="24-hour compound sample")
    sns.lineplot(df_curr.loc[df_curr.variable == "COVID19", :], ax=ax, x="Date", y="value", color=blue, label="Reference", legend=False, alpha=0.6, estimator=np.mean, errorbar=("pi", 90))

    ax2.errorbar(ci_morning.index, ci_morning['mean'], 
                yerr=[ci_morning['mean'] - ci_morning['p5'], ci_morning['p95'] - ci_morning['mean']], 
                fmt='o', color=teal, capsize=4)

    ax2.errorbar(ci_24h.index, ci_24h['mean'], 
                yerr=[ci_24h['mean'] - ci_24h['p5'], ci_24h['p95'] - ci_24h['mean']], 
                fmt='o', color=tan, capsize=4)

    # Set x-labels for bottom row only
    #for ax in axes[-4:]:
    ax.set_xlabel("")

    import matplotlib.dates as mdates

    # Set major ticks every 7 days
    seven_days = mdates.DayLocator(interval=1)
    #for ax in axes:
    ax.xaxis.set_major_locator(seven_days)

    ax.set(ylim=(-100, df_curr.value.max()+100))
    ax2.set(ylim=(-100, df_curr.value.max()+100))
    ax2.set_ylabel(None)
    ax2.set_yticks([])
    ax.set_ylabel('Virus levels\n[copies/l]')

    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2)

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))  # Format without year
    plt.tight_layout()

    # fig.savefig(f"{plot_path}/sampling_strategy_{station}.png", bbox_inches="tight")


In [ ]:
plot_sampling_strategy("S_M2")

### Additional Sampling Strategy Comparisons

In [ ]:
def _compute_ci(df_curr, variable):
    ci = df_curr[df_curr.variable == variable].groupby("Date")["value"].agg(
        mean=lambda x: np.mean(x),
        p5=lambda x: np.percentile(x, 5),
        p95=lambda x: np.percentile(x, 95),
    )
    return ci


def _draw_sampling_strategy_comparison(
    ax,
    station,
    strategy_specs,
    title=None,
    min_date="2020-03-08 08:00:00",
    max_date="2020-03-14 15:00:00",
    show_ylabel=True,
):
    compare_vars = ["COVID19"] + [spec[0] for spec in strategy_specs]
    df_curr = df_sampling.loc[df_sampling.location == station, :]
    df_curr = df_curr.loc[
        (df_curr.Date > min_date)
        & (df_curr.Date < max_date)
        & (df_curr.variable.isin(compare_vars))
    ]

    if df_curr.empty:
        ax.text(0.5, 0.5, f"No data for {station}", ha="center", va="center")
        ax.axis("off")
        return

    ax2 = ax.twinx()

    sns.lineplot(
        df_curr.loc[df_curr.variable == "COVID19", :],
        ax=ax,
        x="Date",
        y="value",
        color=blue,
        label="Reference",
        legend=False,
        alpha=0.6,
        estimator=np.mean,
        errorbar=("pi", 90),
    )

    for variable, label, color, marker in strategy_specs:
        curr = df_curr.loc[df_curr.variable == variable, :]
        if curr.empty:
            print(f"No values found for strategy '{variable}' at {station}.")
            continue

        sns.lineplot(
            curr,
            ax=ax2,
            x="Date",
            y="value",
            errorbar=None,
            color=color,
            marker=marker,
            label=label,
        )

        ci = _compute_ci(df_curr, variable)
        ax2.errorbar(
            ci.index,
            ci["mean"],
            yerr=[ci["mean"] - ci["p5"], ci["p95"] - ci["mean"]],
            fmt=marker,
            color=color,
            capsize=3,
        )

    import matplotlib.dates as mdates

    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.set_xlabel("")

    ymax = df_curr.value.max() + 100
    ax.set(ylim=(-100, ymax))
    ax2.set(ylim=(-100, ymax))
    ax2.set_ylabel(None)
    ax2.set_yticks([])
    if show_ylabel:
        ax.set_ylabel('Virus levels\n[copies/l]')
    else:
        ax.set_ylabel("")

    if title is not None:
        ax.set_title(title)

    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, fontsize=8, frameon=False, loc="upper left")


def _plot_sampling_strategy_comparison(
    station,
    strategy_specs,
    filename,
    title=None,
    min_date="2020-03-08 08:00:00",
    max_date="2020-03-14 15:00:00",
):
    fig, ax = plt.subplots(figsize=(7, 2.7), dpi=300)
    _draw_sampling_strategy_comparison(
        ax,
        station,
        strategy_specs,
        title=title,
        min_date=min_date,
        max_date=max_date,
        show_ylabel=True,
    )
    plt.tight_layout()
    fig.savefig(f"{plot_path}/{filename}_{station}.png", bbox_inches="tight")


def plot_sampling_strategy_15min_30min_vs_24h(station):
    strategy_specs = [
        ("sample_15min_24h", "24-hour compound (15 min)", teal, "o"),
        ("sample_30min_24h", "24-hour compound (30 min)", red, "s"),
        ("24h_sample", "24-hour compound (1h)", tan, "^"),
    ]
    _plot_sampling_strategy_comparison(
        station,
        strategy_specs,
        filename="sampling_strategy_15min_30min_vs_24h",
    )


def plot_sampling_strategy_morning_vs_evening(station):
    strategy_specs = [
        ("morning_sample", "Morning grab (10:00)", teal, "o"),
        ("evening_sample_19", "Evening grab (19:00)", red, "s"),
    ]
    _plot_sampling_strategy_comparison(
        station,
        strategy_specs,
        filename="sampling_strategy_morning_vs_evening",
    )


def plot_sampling_strategy_24h_vs_4h_hourly(station):
    strategy_specs = [
        ("24h_sample", "24-hour compound sample", tan, "o"),
        ("sample_4h_hourly", "4-hour compound (hourly)", red, "s"),
    ]
    _plot_sampling_strategy_comparison(
        station,
        strategy_specs,
        filename="sampling_strategy_24h_vs_4h_hourly",
    )


def plot_sampling_strategies_combined(station, figsize=(16.6, 3.5)):
    comparison_specs = [
        (
            [
                ("sample_15min_24h", "24-hour compound (15 min)", teal, "o"),
                ("sample_30min_24h", "24-hour compound (30 min)", red, "s"),
                ("24h_sample", "24-hour compound (1h)", tan, "^"),
            ],
            "24h: 15min vs 30min vs 1h",
        ),
        (
            [
                ("morning_sample", "Morning grab (10:00)", teal, "o"),
                ("evening_sample_19", "Evening grab (19:00)", red, "s"),
            ],
            "Grab: Morning vs Evening",
        ),
        (
            [
                ("24h_sample", "24-hour compound sample", tan, "o"),
                ("sample_4h_hourly", "4-hour compound (hourly)", red, "s"),
            ],
            "Compound: 24h vs 4h-hourly",
        ),
    ]

    fig, axes = plt.subplots(1, 3, figsize=figsize, dpi=300)
    for idx, (ax, (strategy_specs, title)) in enumerate(zip(axes, comparison_specs)):
        _draw_sampling_strategy_comparison(
            ax,
            station,
            strategy_specs,
            title=title,
            show_ylabel=(idx == 0),
        )

    plt.tight_layout()
    fig.savefig(f"{plot_path}/sampling_strategy_combined_{station}.png", bbox_inches="tight")


In [ ]:
plot_sampling_strategy_15min_30min_vs_24h("S_M2")
plot_sampling_strategy_morning_vs_evening("S_M2")
plot_sampling_strategy_24h_vs_4h_hourly("S_M2")

### Relative Differences Between Sampling Strategies
Calculate mean relative differences for the compared strategy settings.

In [ ]:
def summarize_relative_difference(df_sampling, var_a, var_b, align="day"):
    cols = ["Date", "location", "simulation_id", "variable", "value"]
    df_sub = df_sampling.loc[df_sampling.variable.isin([var_a, var_b]), cols].copy()

    if align == "day":
        df_sub["align_key"] = df_sub["Date"].dt.floor("D")
    elif align == "timestamp":
        df_sub["align_key"] = df_sub["Date"]
    else:
        raise ValueError("align must be 'day' or 'timestamp'")

    df_sub = (
        df_sub.groupby(["align_key", "location", "simulation_id", "variable"]) ["value"]
        .mean()
        .reset_index()
    )

    paired = (
        df_sub.pivot(
            index=["align_key", "location", "simulation_id"],
            columns="variable",
            values="value",
        )
        .dropna(subset=[var_a, var_b])
        .reset_index()
    )

    abs_diff = (paired[var_a] - paired[var_b]).abs()
    denom = paired[var_b].abs().replace(0, np.nan)
    rel_diff_pct = abs_diff / denom * 100

    return {
        "comparison": f"{var_a} vs {var_b}",
        "alignment": align,
        "n_pairs": int(len(paired)),
        "mean_abs_diff": float(abs_diff.mean()),
        "mean_rel_diff_pct": float(rel_diff_pct.mean()),
        "median_rel_diff_pct": float(rel_diff_pct.median()),
        "p95_rel_diff_pct": float(rel_diff_pct.quantile(0.95)),
    }


comparisons = [
    ("sample_15min_24h", "24h_sample", "day"),
    ("sample_30min_24h", "24h_sample", "day"),
    ("sample_15min_24h", "sample_30min_24h", "day"),
    ("morning_sample", "evening_sample_19", "day"),
    ("sample_4h_hourly", "24h_sample", "day"),
]

relative_diff_summary = pd.DataFrame(
    [summarize_relative_difference(df_sampling, a, b, align) for a, b, align in comparisons]
)

relative_diff_summary


In [ ]:
station = "S_M2"
plot_sampling_strategies_combined(station, figsize=(16.6, 3.5))
